# 07 — Create Subscription Orders

The final creation step. For every subscription that got an address in
`06_Create_Addresses.ipynb`:

1. Resolve its plan — `03_Match_Plan_Codes.ipynb` mapping, falling back to
   `STATIC_FALLBACK_PLAN` if unmatched.
2. Optionally attach a primary-contact summary from
   `01_Fetch_Contacts.ipynb` (see `ATTACH_CONTACT_SUMMARY_TO_ORDER`).
3. Build the order payload per the field mapping below.
4. `POST /rest/OrderService/v1/order`.

**vBill -> OneBill field mapping**

| OneBill field | Source |
|---|---|
| Subscription Username | vBill Subscription Label |
| External Service ID | Supplier Service ID |
| Imported Subscription USN | Subscription USN |
| Subscription USN | *(autogenerated by OneBill — not sent)* |
| Radius Username | Voyager `radiusUsers[0]` (from `05_Fetch_Subscriptions.ipynb`'s circuits lookup), falling back to vBill Subscription Label if Voyager didn't return one |
| Activation date | Subscription Start Date |
| Recurring-from date | fixed `2026-08-01` |
| Term | `0` if `SubscriptionEndDate` and `NextPlanStartDate` are both null, else `1` |
| followOnTermDetails.term | (only when Term=1) whole months between today and `SubscriptionEndDate` (falling back to `NextPlanStartDate` if the end date is null) |
| Vendor | `Supplier` mapped via `SUPPLIER_TO_VENDOR` (Chorus/Enable/UFF -> chorus/enable/tff); omitted + logged if `Supplier` is set but unmapped |

> **TODO**: `subscriptionUsername` / `importedSubscriptionUsn` are sent as
> `orderElement` fields by default — flip
> `USE_ATTRIBUTE_STYLE_FOR_USN_FIELDS` in `onebill_common.py` if OneBill
> actually wants them as `orderElementAttribute` entries instead.

> **NOTE**: `resolve_vendor()` reads `subscription["Supplier"]` — this notebook pulls `df_subscriptions` via `SELECT *` from `bi_datastore.billing_subscription`, so double-check that's the exact column name in your MySQL table (vs. e.g. `SupplierName`) before a full run; if it's different, update the `.get("Supplier")` calls in `build_subscription_order_payload`.

## 1. Setup — load everything the previous notebooks produced

In [9]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_subscription_orders")

df_subscriptions   = load_subscriptions_resolved()
df_address_results = load_df("address_results", dtype={"ship_add_id": str})
df_plan_mapping     = try_load_df("plan_mapping", dtype=str)
df_contacts          = try_load_df("contacts")

if df_plan_mapping is None:
    logger.warning("03_plan_code_mapping.csv not found — every subscription falls back to STATIC_FALLBACK_PLAN.")
    df_plan_mapping = pd.DataFrame(columns=["PlanCode", "product_name", "priceplan_name"])

if df_contacts is None and ATTACH_CONTACT_SUMMARY_TO_ORDER:
    logger.warning("01_contacts_by_account.csv not found — orders will be created without a contact summary.")

logger.info(
    f"{len(df_subscriptions):,} subscriptions, {len(df_address_results):,} address results, "
    f"{len(df_plan_mapping):,} plan mappings, "
    f"{len(df_contacts) if df_contacts is not None else 0:,} contact summaries"
)


2026-07-21 08:01:38,422 [INFO] 5 subscriptions, 5 address results, 9 plan mappings, 37,552 contact summaries


## 2. Join subscriptions with their address result

In [10]:
df_work = df_subscriptions.merge(
    df_address_results[["SubscriptionUSN", "status", "ship_add_id", "error"]].rename(
        columns={"status": "AddressStatus", "error": "AddressError"}
    ),
    on="SubscriptionUSN", how="left",
)

ready = df_work[df_work["AddressStatus"].isin(["created", "exists"])]
not_ready = df_work[~df_work["AddressStatus"].isin(["created", "exists"])]
logger.info(f"{len(ready):,} subscriptions have an address and are ready for order creation; "
            f"{len(not_ready):,} do not and will be skipped")
not_ready[["SubscriptionUSN", "AddressStatus", "AddressError"]].head(20)


2026-07-21 08:01:38,446 [INFO] 3 subscriptions have an address and are ready for order creation; 2 do not and will be skipped


,SubscriptionUSN,AddressStatus,AddressError
1,V113062335,skipped,no SupplierServiceID on this subscription
2,V113062343,skipped,no SupplierServiceID on this subscription


## 3. Plan resolution

In [11]:
PLAN_MAPPING = {
    str(row["PlanCode"]).strip().upper(): (row["product_name"], row["priceplan_name"])
    for _, row in df_plan_mapping.iterrows()
}


def resolve_plan(plan_code) -> tuple[str, str, bool]:
    """Returns (productName, priceplanName, matched). matched=False means STATIC_FALLBACK_PLAN was used."""
    key = str(plan_code).strip().upper() if plan_code is not None else None
    if key in PLAN_MAPPING:
        product_name, priceplan_name = PLAN_MAPPING[key]
        return product_name, priceplan_name, True
    return STATIC_FALLBACK_PLAN["productName"], STATIC_FALLBACK_PLAN["priceplanName"], False


## 4. Contact-summary lookup (optional)

Joins each subscription's *original* vBill `AccountCode` (not the bucket
account) against `01_Fetch_Contacts.ipynb`'s output.

In [12]:
CONTACT_BY_ACCOUNT = (
    {str(row["AccountCode"]): row for _, row in df_contacts.iterrows()}
    if df_contacts is not None else {}
)


def contact_summary_attributes(account_code) -> list[dict]:
    if not ATTACH_CONTACT_SUMMARY_TO_ORDER:
        return []
    contact = CONTACT_BY_ACCOUNT.get(str(account_code))
    if contact is None:
        return []
    attrs = []
    if clean(contact.get("ContactName")):
        attrs.append({"featureName": "Original Contact Name", "value": contact["ContactName"]})
    if clean(contact.get("ContactEmail")):
        attrs.append({"featureName": "Original Contact Email", "value": contact["ContactEmail"]})
    if clean(contact.get("ContactPhone")):
        attrs.append({"featureName": "Original Contact Phone", "value": str(contact["ContactPhone"])})
    return attrs


## 5. Build the order payload (field mapping)

In [ ]:
def build_subscription_order_payload(subscription: dict, ship_add_id: str, product_name: str, priceplan_name: str) -> dict:
    # clean() turns NaN/blank into None so the fallbacks below actually fire — NaN is
    # truthy in Python, so an un-cleaned NaN silently skips an "or" fallback and ends up
    # in the JSON payload, which `requests` then refuses to serialize
    # ("Out of range float values are not JSON compliant: nan"). Same class of bug as the
    # one fixed earlier in add_address_to_account.
    quantity = clean(subscription.get("Quantity"))
    quantity = quantity if quantity is not None else DEFAULT_QUANTITY

    supplier_service_id = clean(subscription.get("SupplierServiceID"))

    radius_username = clean(subscription.get("ParsedAddress_radius_user")) or clean(subscription.get("SubscriptionLabel"))

    # --- term / follow-on term -------------------------------------------------
    # term = 0 (no follow-on) unless the subscription has a SubscriptionEndDate
    # or a NextPlanStartDate, in which case term = 1 and OneBill needs a
    # followOnTermDetails block whose `term` is the whole number of months
    # between today and the subscription's end date.
    subscription_end_date = clean(subscription.get("SubscriptionEndDate"))
    next_plan_start_date = clean(subscription.get("NextPlanStartDate"))
    has_follow_on_term = subscription_end_date is not None or next_plan_start_date is not None

    order_element = {
        "quantity":               quantity,
        "actionType":             DEFAULT_ACTION_TYPE,
        "fulfilledDate":          to_iso_midnight(subscription["SubscriptionStartDate"]),  # activation date
        "recurringStartDate":     RECURRING_FROM_DATE,                                       # fixed 2026-08-01
        "productName":            product_name,
        "priceplanName":          priceplan_name,
        "shipAddId":              ship_add_id,
        "term":                   1 if has_follow_on_term else 0,
    }

    if has_follow_on_term:
        # Prefer SubscriptionEndDate as the reference date for the month count;
        # fall back to NextPlanStartDate if only that is populated.
        follow_on_reference_date = subscription_end_date or next_plan_start_date
        follow_on_months = months_between(datetime.now(), follow_on_reference_date)

        order_element["termAction"] = 1
        order_element["followOnTermDetails"] = {
            "term":         follow_on_months,
            "termMode":     "M",
            "termAction":   1,
            "termAligned":  False,
            "termType":     0,
        }

    order_element_attributes = [
        {"featureName": "Radius Username", "type": "0", "value": str(radius_username)},
        {"featureName": "External Service ID", "value": supplier_service_id},
    ]

    vendor = resolve_vendor(subscription.get("Supplier"))
    if vendor is not None:
        order_element_attributes.append({"featureName": "Vendor", "value": vendor})
    elif clean(subscription.get("Supplier")) is not None:
        # Supplier is populated but isn't Chorus/Enable/UFF — flag it rather than
        # silently omitting the Vendor attribute. See SUPPLIER_TO_VENDOR in onebill_common.py.
        logger.warning(
            f"subscription {subscription.get('SubscriptionUSN')} has an unmapped Supplier "
            f"({subscription.get('Supplier')!r}) — order created without a Vendor attribute"
        )

    if USE_ATTRIBUTE_STYLE_FOR_USN_FIELDS:
        order_element_attributes.extend([
            {"featureName": "Subscription Username", "value": str(subscription["SubscriptionLabel"])},
            {"featureName": "Imported Subscription USN", "value": str(subscription["SubscriptionUSN"])},
        ])
    else:
        order_element["subscriptionUsername"] = str(subscription["SubscriptionLabel"])
        order_element["importedSubscriptionUsn"] = str(subscription["SubscriptionUSN"])

    order_element_attributes.extend(contact_summary_attributes(subscription.get("AccountCode")))

    # orderElementAttribute lives INSIDE the orderElement item, not as a sibling
    # key on the outer payload — confirmed against the working Postman example.
    order_element["orderElementAttribute"] = order_element_attributes

    return {
        "accountNumber":      str(subscription["TargetAccountNumber"]),
        "orderState":         DEFAULT_ORDER_STATE,
        "billThissOrder":     False,
        "isSkipProvisioning": True,
        "orderElement":       [order_element],
    }


## 6. Per-subscription worker

In [14]:
def create_order_for_subscription(session: requests.Session, subscription: dict) -> dict:
    subscription_id = subscription["SubscriptionUSN"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  subscription["TargetAccountNumber"],
        "status":               "failed",
        "plan_matched":         None,
        "productName":          None,
        "priceplanName":        None,
        "onebill_order_id":     None,
        "error":                None,
    }

    try:
        product_name, priceplan_name, matched = resolve_plan(subscription.get(SUBSCRIPTION_PLANCODE_COLUMN))
        result["plan_matched"]  = matched
        result["productName"]   = product_name
        result["priceplanName"] = priceplan_name

        payload = build_subscription_order_payload(
            subscription, subscription["ship_add_id"], product_name, priceplan_name
        )
        response = create_onebill_order(session, payload)

        result["status"] = "success"
        result["onebill_order_id"] = response.get("orderId", "unknown")
        logger.info(f"[OK] subscription {subscription_id} -> {subscription['TargetAccountNumber']} "
                    f"(plan_matched={matched}, orderId={result['onebill_order_id']})")

    except Exception as e:
        result["error"] = str(e)
        logger.error(f"[FAIL] subscription {subscription_id} — {e}")

    return result


## 7. Run (parallel driver)

In [15]:
def create_all_orders(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []

    logger.info(f"Creating orders for {total:,} subscriptions with {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_order_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "success")
                logger.info(f"Progress: {i}/{total} — {ok} succeeded so far")

    return pd.DataFrame(results)


order_results_df = create_all_orders(ready)
order_results_df.head(20)


2026-07-21 08:01:40,511 [INFO] Creating orders for 3 subscriptions with 10 workers...
2026-07-21 08:01:46,777 [INFO] [OK] subscription V113062392 -> 99965692041234 (plan_matched=True, orderId=unknown)
2026-07-21 08:01:48,294 [INFO] [OK] subscription V113061451 -> 99965692041234 (plan_matched=True, orderId=unknown)
2026-07-21 08:01:49,877 [INFO] [OK] subscription V113062384 -> 99965692041234 (plan_matched=True, orderId=unknown)
2026-07-21 08:01:49,880 [INFO] Progress: 3/3 — 3 succeeded so far


,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error
0,V113062392,99965692041234,success,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,unknown,None
1,V113061451,99965692041234,success,True,Wholesale Fibre BS2 (Enable),WS Tail+Data - BS2 Res (Enable) - 920/500,unknown,None
2,V113062384,99965692041234,success,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,unknown,None


## 8. Save

In [16]:
save_df("order_results", order_results_df)


Saved 3 rows -> migration_data\07_order_creation_results.csv
